In [0]:
!pip install beautifulsoup4 requests demjson3 lxml

In [0]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag # Import Tag explicitly
from lxml import etree, html # Import html for parsing in lxml
import json
import demjson3
from datetime import datetime, timedelta
import pandas as pd
from zoneinfo import ZoneInfo

In [0]:
arg_time = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires"))
base_volume = "/Volumes/workspace/futbol/futbol_landing"
current_timestamp = (arg_time - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Timestamp actual: {current_timestamp}")

In [0]:
def scrape_html_content(url, html_selector, selector_type=None, multiple=False):
    """
    Scrapes content from a given URL based on HTML selector and selector type.

    Args:
        url (str): The URL to fetch HTML content from.
        html_selector (str): The selector string (e.g., 'div.content', '//h1', 'p').
        selector_type (str): Type of selector: 'css_selector', 'xpath', 'tag_name', 'class'.
                             Defaults to 'css_selector'.
        multiple (bool): If True, returns a list of all matching elements.
                         If False, returns the first matching element. Defaults to False.
    Returns:
        list or object: A list of scraped contents/elements if 'multiple' is True,
                        or a single content/element if 'multiple' is False.
                        Returns None or empty list on failure or no matches.
    """
    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    except requests.exceptions.RequestException as e:
        print(f"Error accessing URL {url}: {e}")
        return [] if multiple else None

    html_content = response.text
    found_elements = []

    if selector_type == 'xpath':
        # Use lxml for both parsing and selection if selector_type is xpath
        try:
            tree = html.fromstring(html_content)
            found_elements = tree.xpath(html_selector)
            # Removed: print(f"LIST OF ELEMENTS FOUNDED: {found_elements} BY XPATH")
        except Exception as e:
            print(f"Error parsing with lxml or executing XPath: {e}")
            return [] if multiple else None
    else:
        # Use BeautifulSoup for other selector types
        soup = BeautifulSoup(html_content, 'html.parser')
        if selector_type == 'css_selector':
            found_elements = soup.select(html_selector)
        elif selector_type == 'tag_name':
            found_elements = soup.find_all(html_selector)
        elif selector_type == 'class':
            # html_selector should be just the class name here
            found_elements = soup.find_all(class_=html_selector)
        else:
            print(f"Unsupported selector_type: {selector_type} for BeautifulSoup.")
            return [] if multiple else None

    if not found_elements:
        print(f"No elements found for selector '{html_selector}' with type '{selector_type}'.")
        return [] if multiple else None

    results = []
    # Removed: print(f"LIST OF ELEMENTS FOUNDED: {found_elements}")
    # Process each found element
    for element in found_elements:
        extracted_content = None
        # Removed: print(f"ELEMENT FOUNDED: {element}")

        # Ensure we are working with an element object, not raw text from XPath /text()
        if isinstance(element, (Tag, html.HtmlElement)):
            tag_name = element.name if isinstance(element, Tag) else getattr(element, 'tag', '').lower()
            element_type = element.get('type') if isinstance(element, Tag) else element.get('type')
            data_react_helmet = element.get('data-react-helmet') if isinstance(element, Tag) else element.get('data-react-helmet')

            if tag_name == 'script' and element_type == 'application/ld+json' and data_react_helmet == 'true':
                script_content = element.string if isinstance(element, Tag) else element.text_content()
                if script_content:
                    try:
                        extracted_content = json.loads(script_content)
                    except json.JSONDecodeError:
                        try:
                            extracted_content = demjson3.decode(script_content)
                        except demjson3.JSONDecodeError as e:
                            print(f"Warning: Could not parse JSON-LD with demjson3. Error: {e}. Content snippet: {script_content[:100]}...")
                            extracted_content = None
                else:
                    extracted_content = None
            else:
                if isinstance(element, Tag):
                    extracted_content = element.get_text(strip=True)
                elif isinstance(element, html.HtmlElement):
                    extracted_content = element.xpath('string()').strip()
                else:
                    extracted_content = str(element).strip()
        else:
            # This case handles when XPath returns raw text directly (e.g., using /text() in XPath)
            # Attempt to parse as JSON if it looks like JSON, otherwise keep as string
            try:
                extracted_content = json.loads(element)
            except (json.JSONDecodeError, TypeError):
                try:
                    extracted_content = demjson3.decode(element)
                except (demjson3.JSONDecodeError, TypeError):
                    extracted_content = str(element).strip()

        results.append(extracted_content)

    if multiple:
        # print(f"RESULTS ----->\n{results}")
        return results
    else:
        return results[0] if results else None

In [0]:
def run(date):
    matchs_scripts = scrape_html_content(
        f"https://canchallena.lanacion.com.ar/fecha/{date}/",
        '//script[@type="application/ld+json" and @data-react-helmet="true"]',
        selector_type="xpath",
        multiple=True
    )
    current_matches = []
    for match in matchs_scripts:
        if match.get("@type", "") == "SportsEvent":
            name = match.get("name", "")
            link = match.get("url", "")
            start_date = match.get("startDate", "")
            league = match.get("organizer", {}).get("name", "")
            if "https://canchallena.lanacion.com.ar/futbol" in link:
                match_details = scrape_html_content(
                    link,
                    '//script[@type="application/ld+json" and @data-react-helmet="true"]',
                    selector_type="xpath",
                    multiple=True
                )
                current_match = {
                    "scrape_date": date,
                    "date": start_date,
                    "name": name,
                    "league": league,
                    "link": link,
                    "match": match_details
                }
                current_matches.append(current_match)
    year, month, day = date.split("-")
    path = f"{base_volume}/raw/year={year}/month={month}/day={day}"
    dbutils.fs.mkdirs(path)
    df_current_match = pd.json_normalize(current_matches, max_level=1)
    print(f"DF CURRENT MATCH -----> \n{df_current_match.head()} -----> \n{df_current_match.info()}")
    full_path = f"{path}/{date}.parquet"
    df_current_match.to_parquet(full_path, index=False)
    print(f"✅ Guardado: {full_path}")
    print("")
    print("")
    print("")

In [0]:
run(current_timestamp)

In [0]:
# from datetime import datetime, timedelta

# start_date = datetime(2024, 1, 1)
# end_date = datetime(2024, 12, 31)
# # end_date = datetime.now() - timedelta(days=1)

# date_range = []
# current = start_date
# while current <= end_date:
#     date_range.append(current.strftime("%Y-%m-%d")[0:10])
#     current += timedelta(days=1)

# for date_str in date_range:
#     # Aquí puedes llamar a tu función o procesar cada fecha
#     print(date_str)
#     run(date_str)